# 03 Baseline Comparison

Compare three generation strategies for the Autonomous Research Assistant: a plain pretrained `flan-t5-base` prompt, a stronger prompt-engineered baseline, and a RAG-grounded prompt using Chroma retrieval.

## Setup

The notebook uses a small evaluation subset so it can run locally while still producing real model outputs and real metrics.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Could not locate project root containing src/')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import evaluate_generation, save_metrics
from src.rag_pipeline import RagGenerator, build_review_prompt, format_retrieved_context
from src.vector_store import ChromaVectorStore, load_embedding_model

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
FINETUNE_TEST_JSONL = DATA_DIR / 'finetune_test.jsonl'
CHROMA_DIR = DATA_DIR / 'chroma'
METRICS_CSV = OUTPUTS_DIR / 'metrics.csv'
GENERATIONS_CSV = OUTPUTS_DIR / 'baseline_generations.csv'
COMPARISON_MD = OUTPUTS_DIR / 'baseline_comparison.md'

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_colwidth', 160)
print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/Lenovo/Desktop/sem 6/Gen_AI Project


## Load Evaluation Examples

The references come from the instruction-style dataset created during data collection. The selected tasks match the app's core outputs.

In [2]:
if not FINETUNE_TEST_JSONL.exists():
    raise FileNotFoundError('Missing data/finetune_test.jsonl. Run 01_data_collection.ipynb first.')
if not (CHROMA_DIR / 'chroma.sqlite3').exists():
    raise FileNotFoundError('Missing Chroma index. Run 02_vector_database.ipynb first.')

with FINETUNE_TEST_JSONL.open('r', encoding='utf-8') as handle:
    test_records = [json.loads(line) for line in handle]

test_df = pd.DataFrame(test_records)
test_df.groupby('task').size().reset_index(name='examples')

,task,examples
0,comparative_analysis,7
1,evidence_based_qa,47
2,literature_review,6
3,paper_summary,51
4,research_gap_analysis,9
5,technical_explanation,37


In [3]:
preferred_tasks = ['literature_review', 'research_gap_analysis', 'technical_explanation']
eval_examples = []

for task in preferred_tasks:
    task_rows = test_df[test_df['task'] == task]
    if not task_rows.empty:
        eval_examples.append(task_rows.iloc[0].to_dict())

if len(eval_examples) < 3:
    remaining = test_df[~test_df.index.isin([test_df.index[test_df.eq(example).all(axis=1)][0] for example in eval_examples])]
    eval_examples.extend(remaining.head(3 - len(eval_examples)).to_dict(orient='records'))

eval_df = pd.DataFrame(eval_examples).reset_index(drop=True)
eval_df[['task', 'topic', 'instruction', 'output']]

,task,topic,instruction,output
0,literature_review,RAG for citation-grounded text generation,Generate a short literature review on RAG for citation-grounded text generation using the provided papers.,Research on RAG for citation-grounded text generation shows a clear focus on methods represented by papers such as From RAG to QA-RAG: Integrating Generativ...
1,research_gap_analysis,Open-source large language models,Identify research gaps for Open-source large language models using the provided papers.,"For Open-source large language models, the retrieved papers suggest several research gaps: more reliable evaluation protocols, stronger evidence grounding, ..."
2,technical_explanation,Neural information retrieval for scholarly search,Explain the paper's main idea in simple terms for an early-stage researcher.,"In simple terms, this paper is about Ranking models are the main components of information retrieval systems. The main value is that it helps readers unders..."


## Load Models and Retrieval Store

The same `flan-t5-base` model is reused across all three strategies. The only difference is the prompt and whether retrieved evidence is included.

In [4]:
generator = RagGenerator('google/flan-t5-base')
embedding_model = load_embedding_model()
vector_store = ChromaVectorStore(str(CHROMA_DIR))

print(f'Chroma indexed chunks: {vector_store.collection.count()}')

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chroma indexed chunks: 928


## Prompt Builders

The pretrained baseline receives a minimal instruction, the prompt-engineered baseline receives a structured instruction, and the RAG baseline receives retrieved evidence.

In [5]:
MODE_BY_TASK = {
    'literature_review': 'literature_review',
    'research_gap_analysis': 'gap_analysis',
    'technical_explanation': 'technical_explanation',
    'evidence_based_qa': 'qa',
    'paper_summary': 'technical_explanation',
    'comparative_analysis': 'literature_review',
}

def build_plain_prompt(example: dict) -> str:
    return f"{example['instruction']}\n\n{example['input']}"


def build_prompt_engineered_prompt(example: dict) -> str:
    return (
        'You are an academic research assistant. Produce a concise, evidence-aware answer.\n'
        'Use only the provided input. Avoid unsupported claims.\n\n'
        f"Task: {example['task']}\n"
        f"Topic: {example['topic']}\n"
        f"Instruction: {example['instruction']}\n"
        f"Input:\n{example['input']}\n\n"
        'Answer:'
    )


def build_rag_prompt(example: dict, top_k: int = 5) -> tuple[str, dict]:
    query = f"{example['topic']} {example['instruction']}"
    retrieval_results = vector_store.semantic_search(query, embedding_model, top_k=top_k)
    context = format_retrieved_context(retrieval_results)
    mode = MODE_BY_TASK.get(example['task'], 'literature_review')
    prompt = build_review_prompt(example['topic'], context, mode=mode)
    prompt += f"\n\nUser instruction: {example['instruction']}\nAnswer:"
    return prompt, retrieval_results


print(build_prompt_engineered_prompt(eval_examples[0])[:700])

You are an academic research assistant. Produce a concise, evidence-aware answer.
Use only the provided input. Avoid unsupported claims.

Task: literature_review
Topic: RAG for citation-grounded text generation
Instruction: Generate a short literature review on RAG for citation-grounded text generation using the provided papers.
Input:
Paper 1: From RAG to QA-RAG: Integrating Generative AI for Pharmaceutical Regulatory Compliance Process. Abstract: Regulatory compliance in the pharmaceutical industry entails navigating through complex and voluminous guidelines, often requiring significant human resources. To address these challenges, our study introduces a chatbot model that utilizes generat


## Generate Outputs

This cell writes the raw generations so the report can include qualitative examples, not only metric tables.

In [6]:
generation_rows = []

for example_id, example in enumerate(eval_examples, start=1):
    strategies = [
        ('pretrained', build_plain_prompt(example), None),
        ('prompt_engineered', build_prompt_engineered_prompt(example), None),
    ]
    rag_prompt, retrieval_results = build_rag_prompt(example, top_k=5)
    strategies.append(('rag_system', rag_prompt, retrieval_results))

    for strategy, prompt, retrieval_results in strategies:
        print(f"Generating example {example_id} with {strategy}...")
        candidate = generator.generate(prompt, max_new_tokens=128)
        retrieved_titles = []
        if retrieval_results is not None:
            retrieved_titles = [meta.get('title', '') for meta in retrieval_results.get('metadatas', [[]])[0]]

        generation_rows.append(
            {
                'example_id': example_id,
                'strategy': strategy,
                'task': example['task'],
                'topic': example['topic'],
                'instruction': example['instruction'],
                'reference': example['output'],
                'candidate': candidate,
                'retrieved_titles': ' | '.join(retrieved_titles),
            }
        )

generations_df = pd.DataFrame(generation_rows)
generations_df.to_csv(GENERATIONS_CSV, index=False)
generations_df[['example_id', 'strategy', 'task', 'candidate']]


Generating example 1 with pretrained...


Generating example 1 with prompt_engineered...


Generating example 1 with rag_system...


Generating example 2 with pretrained...


Generating example 2 with prompt_engineered...


Generating example 2 with rag_system...


Generating example 3 with pretrained...


Generating example 3 with prompt_engineered...


Generating example 3 with rag_system...


,example_id,strategy,task,candidate
0,1,pretrained,literature_review,Abstract: The authors propose a collaborative training framework for RAG for open domain question answering.
1,1,prompt_engineered,literature_review,We introduce a chatbot model that utilizes generative AI and the Retrieval Augmented Generation method.
2,1,rag_system,literature_review,Retrieval Augmented Generation: A Framework for Reference-Free Evaluation of Retrieval Augmented Generation
3,2,pretrained,research_gap_analysis,We propose a new approach to self-cognition in large language models.
4,2,prompt_engineered,research_gap_analysis,We propose a new approach to self-cognition in large language models.
5,2,rag_system,research_gap_analysis,"Open-source large language models are a promising approach to re-engineering human coders, they are also vulnerable to re-engineering and re-learning."
6,3,pretrained,technical_explanation,We compare the proposed neural ranking models in the literature and propose future research directions.
7,3,prompt_engineered,technical_explanation,Neural ranking models for document retrieval
8,3,rag_system,technical_explanation,"Neural information retrieval is a method of retrieving information from a corpus of documents, e.g., a document, a answer, a video, a document, a page, a pa..."


## Evaluate Generations

BLEU and ROUGE measure lexical overlap with the reference. BERTScore gives a semantic similarity signal; this notebook uses a smaller model to keep the run practical on local hardware.

In [7]:
BERTSCORE_MODEL = 'distilbert-base-uncased'
BERTSCORE_NUM_LAYERS = 5

metric_rows = []
for row in generations_df.itertuples(index=False):
    scores = evaluate_generation(
        row.reference,
        row.candidate,
        bertscore_model=BERTSCORE_MODEL,
        bertscore_num_layers=BERTSCORE_NUM_LAYERS,
    )
    scores.update(
        {
            'model': row.strategy,
            'example_id': row.example_id,
            'task': row.task,
            'topic': row.topic,
        }
    )
    metric_rows.append(scores)

metrics_df = save_metrics(metric_rows, METRICS_CSV)
metrics_df[['model', 'example_id', 'task', 'bleu', 'rouge1', 'rouge2', 'rougeL', 'bertscore_f1']]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,model,example_id,task,bleu,rouge1,rouge2,rougeL,bertscore_f1
0,pretrained,1,literature_review,0.000136,0.144000,0.032520,0.096000,0.753285
1,prompt_engineered,1,literature_review,0.000076,0.176000,0.048780,0.112000,0.768469
2,rag_system,1,literature_review,0.000016,0.162602,0.082645,0.113821,0.770020
3,pretrained,2,research_gap_analysis,0.001655,0.095238,0.065574,0.095238,0.771482
4,prompt_engineered,2,research_gap_analysis,0.001655,0.095238,0.065574,0.095238,0.771482
5,rag_system,2,research_gap_analysis,0.013958,0.186667,0.109589,0.186667,0.787114
6,pretrained,3,technical_explanation,0.004034,0.226415,0.039216,0.150943,0.776770
7,prompt_engineered,3,technical_explanation,0.000234,0.136364,0.047619,0.136364,0.762130
8,rag_system,3,technical_explanation,0.007448,0.115385,0.019608,0.096154,0.705889


## Aggregate Results

The aggregate table is the main quantitative result for the baseline-comparison section of the final report.

In [8]:
aggregate_metrics = (
    metrics_df.groupby('model')[['bleu', 'rouge1', 'rouge2', 'rougeL', 'bertscore_f1']]
    .mean()
    .reset_index()
    .sort_values('bertscore_f1', ascending=False)
)
aggregate_metrics

,model,bleu,rouge1,rouge2,rougeL,bertscore_f1
1,prompt_engineered,0.000655,0.135867,0.053991,0.114534,0.767360
0,pretrained,0.001942,0.155218,0.045770,0.114060,0.767179
2,rag_system,0.007141,0.154884,0.070614,0.132214,0.754341


## Save Report Summary

This Markdown file keeps the key outputs in a compact format for the final write-up.

In [9]:
def dataframe_to_markdown_table(df: pd.DataFrame) -> str:
    headers = list(df.columns)
    table_lines = [
        '| ' + ' | '.join(headers) + ' |',
        '| ' + ' | '.join(['---'] * len(headers)) + ' |',
    ]
    for row in df.itertuples(index=False):
        values = []
        for value in row:
            if isinstance(value, float):
                values.append(f'{value:.4f}')
            else:
                values.append(str(value))
        table_lines.append('| ' + ' | '.join(values) + ' |')
    return '\n'.join(table_lines)


lines = ['# Baseline Comparison Results', '']
lines.append('## Aggregate Metrics')
lines.append(dataframe_to_markdown_table(aggregate_metrics))
lines.append('')

for example_id, group in generations_df.groupby('example_id'):
    first = group.iloc[0]
    lines.append(f"## Example {example_id}: {first['task']}")
    lines.append(f"Topic: {first['topic']}")
    lines.append('')
    lines.append('Reference:')
    lines.append(first['reference'])
    lines.append('')
    for row in group.itertuples(index=False):
        lines.append(f"### {row.strategy}")
        lines.append(row.candidate)
        if row.retrieved_titles:
            lines.append('')
            lines.append(f"Retrieved titles: {row.retrieved_titles}")
        lines.append('')

COMPARISON_MD.write_text('\n'.join(lines), encoding='utf-8')
print(f'Wrote {COMPARISON_MD.relative_to(PROJECT_ROOT)}')

Wrote outputs/baseline_comparison.md


## Final Artifact Check

In [10]:
artifact_paths = [METRICS_CSV, GENERATIONS_CSV, COMPARISON_MD]
pd.DataFrame(
    {
        'artifact': [str(path.relative_to(PROJECT_ROOT)) for path in artifact_paths],
        'exists': [path.exists() for path in artifact_paths],
        'size_bytes': [path.stat().st_size if path.exists() else 0 for path in artifact_paths],
    }
)

,artifact,exists,size_bytes
0,outputs/metrics.csv,True,1646
1,outputs/baseline_generations.csv,True,8141
2,outputs/baseline_comparison.md,True,4535
